#  Instruction Level Parallelism


Enter your name and student ID.

 * Name:
 * Student ID:


# 1. Introduction
* we have learned that CPUs have multicore and SIMD parallelism
* the last dimension of parallelism is _Instruction Level Parallelism (ILP)_, the ability to execute many instructions <font color=red>of a single thread</font> concurrently (i.e., the executions of many instructions overlap in time)
* in contrast, GPUs do not aggressively try to extract ILP from a single thread; their parallelism mostly comes from executing many threads simultaneously
* illuminating the difference between the two is both instructive and practically important, especially for optimizing CPU code

# 2. How to work on this notebook
* read text, modify cells if necessary, and execute them
* many cells can be executed without modifying them
* cells that need your work are typically preceded by <font color="green">text colored green explaining what you need to do</font>
* problems with non-trivial work have a section header starting with <font color="green">"Problem X :"</font>

# 3. Jupyter AI
* you can ask Jupyter AI when you need help
* open [pd08_ilp.chat](pd08_ilp.chat) in this folder
* choose `OpenCode` in Persona Chooser
* you can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* it is instructed to help you but not to give you answers to exercise problems directly; do not try to circumvent it

# 4. Setting up compilers
- the setup is the same as the one we used for OpenMP


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

# 5. Setting up the extension to submit jobs
- extension to use `%%sub` magic cell


In [ ]:
%load_ext miyabi

# 6. CPU without ILP
* this is an experiment very similar to the one we did on GPUs
* each thread repeats x = a * x + b many times and occasionally records the time
* although the primary focus is single-thread performance, the program is written with OpenMP so that it can be executed on multicore CPUs and GPUs
* the compilation option below enables execution on both GPU and CPU

* the program consists of three files
  * `ilp.h` : declarations shared by the other two
  * `ilp_main.cc` : `main`, the cycle counter, etc.; you do not have to read it
  * `no_ilp.cc` (and later `simd.cc`, `simd2.cc`, `simd_ilp.cc`, ...) : only the function `iter_fun`, which does the computation we are interested in
* `iter_fun` is put in a separate file so that its assembly code (which we will look at later) contains nothing else


In [ ]:
%%writefile ilp.h
/* declarations shared by ilp_main.cc and the file defining iter_fun */
typedef struct {
  double x;
  int core[2];
} record_t;

/* get_clock and get_core are defined in ilp_main.cc and iter_fun in another file.
   "declare target" makes them callable on GPU too (needed by clang and nvc++
   because they are defined in a different file from the target region;
   gcc, which we use only for CPU, rejects get_cpu_cycles under it) */
#if defined(__clang__) || defined(__NVCOMPILER)
#pragma omp declare target
#endif
long get_clock(void);
unsigned int get_core(void);
void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T);
#if defined(__clang__) || defined(__NVCOMPILER)
#pragma omp end declare target
#endif

In [ ]:
%%writefile ilp_main.cc
/* everything except iter_fun (which is in a separate file
   so that its assembly is easy to read) */
#include <assert.h>
#include <err.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>
#include <sched.h>
#include <stdint.h>
#include <string.h>
#include <unistd.h>
#include <sys/syscall.h>
#include <linux/perf_event.h>
#include "ilp.h"

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}


/* get the number of CPU core cycles this thread has executed so far
   (counted by the hardware performance counter, via perf_event_open).
   unlike rdtsc on x86 or cntvct_el0 on Arm, which tick at a constant rate
   regardless of the actual core frequency, this counts real core cycles.
   requires /proc/sys/kernel/perf_event_paranoid <= 2 */
static __thread int perf_fd = -1;  /* one counter per thread */
#if __NVCOMPILER
__host__                        /* CPU only; never compiled for GPU */
#endif
__attribute__((unused))
static long get_cpu_cycles(void) {
  int fd = perf_fd;
  if (fd == -1) {
    struct perf_event_attr pe;
    memset(&pe, 0, sizeof(pe));
    pe.type = PERF_TYPE_HARDWARE;
    pe.size = sizeof(pe);
    pe.config = PERF_COUNT_HW_CPU_CYCLES;
    pe.exclude_kernel = 1;      /* count only user-space cycles */
    pe.exclude_hv = 1;
    /* pid = 0, cpu = -1 : this thread, on whichever core it runs */
    fd = syscall(SYS_perf_event_open, &pe, 0, -1, -1, 0);
    if (fd == -1) err(1, "perf_event_open");
    perf_fd = fd;
  }
  long long c;
  if (read(fd, &c, sizeof(c)) != sizeof(c)) err(1, "read");
  return c;
}

#if __NVCOMPILER
#include <nv/target>
/* get core number (SM id for GPU). */
__host__ __device__ unsigned int get_core(void) {
  if target(nv::target::is_device) {
    unsigned int sm;
    asm("mov.u32 %0, %%smid;" : "=r"(sm));
    return sm;
  } else {
    return sched_getcpu();
  }
}

/* get GPU/CPU clock (for Clang LLVM compiler) */
long get_clock(void) {
  if target(nv::target::is_device) {
    long clock;
    asm volatile("mov.s64 %0, %%clock64;" : "=l" (clock));
    return clock;
  } else {
    return get_cpu_cycles();
  }
}

#else  // __clang__ or GCC
/* get SM id (for Clang LLVM compiler).
   return -1 if called on CPU */
unsigned int get_core(void) {
#if __CUDA_ARCH__
  unsigned int sm;
  asm("mov.u32 %0, %%smid;" : "=r"(sm));
  return sm;
#else
  return sched_getcpu();
#endif
}

/* get GPU/CPU clock (for Clang LLVM compiler) */
long get_clock(void) {
#if __CUDA_ARCH__
  long clock;
  asm volatile("mov.s64 %0, %%clock64;" : "=l" (clock));
  return clock;
#else
  return get_cpu_cycles();
#endif
}
#endif

void dump(record_t * R, long * T, long L, long M, long N) {
  long k = 0;
  for (long i = 0; i < L; i++) {
    /* T[k], ..., T[k+M-1] are M time stamps = (M - 1) intervals of N iterations each */
    long dt = T[k + M - 1] - T[k];
    double avg = (double)dt / ((double)(M - 1) * (double)N);
    printf("i=%ld x=%f core0=%d core1=%d cycles_per_iter=%f",
           i, R[i].x, R[i].core[0], R[i].core[1], avg);
    for (long j = 0; j < M; j++) {
      printf(" %ld", T[k]);
      k++;
    }
    printf("\n");
  }
}

int getenv_int(const char * v) {
  char * s = getenv(v);
  if (!s) {
    fprintf(stderr, "specify environment variable %s\n", v);
    exit(1);
  }
  return atoi(s);
}

int main(int argc, char ** argv) {
  int idx = 1;
  long L   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long M   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  if (M < 2) { fprintf(stderr, "M must be >= 2\n"); exit(1); }
  long N   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  float a = (idx < argc ? atof(argv[idx]) : 0.99); idx++;
  float b = (idx < argc ? atof(argv[idx]) : 1.00); idx++;
  int n_teams = getenv_int("OMP_NUM_TEAMS");
  int n_threads_per_team = getenv_int("OMP_NUM_THREADS");
  record_t * R = (record_t *)calloc(L, sizeof(record_t));
  long * T = (long *)calloc(L * M, sizeof(long));
#pragma omp target teams distribute parallel for num_teams(n_teams) num_threads(n_threads_per_team) map(tofrom: R[0:L], T[0:L*M])
  for (long i = 0; i < L; i++) {
    iter_fun(a, b, i, M, N, R, T);
  }
  dump(R, T, L, M, N);
  return 0;
}


In [ ]:
%%writefile no_ilp.cc
#include "ilp.h"

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
  float x = i;
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      x = a * x + b;
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  R[i].x = x;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp no_ilp.cc ilp_main.cc -o no_ilp
#nvc++   -Wall -O3 -tp neoverse-v2 -mp=multicore -cuda no_ilp.cc ilp_main.cc -o no_ilp
#g++     -Wall -O3 -mcpu=neoverse-v2 -fopenmp no_ilp.cc ilp_main.cc -o no_ilp

* run it on a single core


In [ ]:
%%bash
OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./no_ilp 1 100 1000000 > a.dat
cat a.dat

* it shows the cycles per iteration (of the innermost loop) as `cycles_per_iter=xxxx`, which can be extracted by


In [ ]:
%%bash
awk '{print $5}' a.dat

* this is essentially the _latency_ of an FMA instruction 
* it should come close to 4, the latency of an FMA instruction according to the [Arm Neoverse V2 Software Optimization Guide](https://developer.arm.com/documentation/PJDOC-466751330-593177/latest/)
* note on how we measure cycles (see `get_cpu_cycles` in `ilp_main.cc`):
  * we count the actual core cycles with the hardware performance counter of the CPU, accessed via the Linux `perf_event_open` system call (the same mechanism the `perf` command uses)
  * the more familiar "clock" instructions (`rdtsc` on x86 or `cntvct_el0` on Arm) do _not_ count core cycles; they tick at a constant rate regardless of the actual core frequency (1GHz on Grace, whereas the core runs at about 3.3GHz), so they essentially give you the time in a fixed unit

* let's look at the instruction sequence
* identify the instruction sequence corresponding to the innermost loop


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp no_ilp.cc -S
#nvc++   -Wall -O3 -tp neoverse-v2 -mp=multicore -cuda no_ilp.cc -S
#g++     -Wall -O3 -mcpu=neoverse-v2 -fopenmp no_ilp.cc -S

In [ ]:
%%bash
cat no_ilp.s

* you can also visualize the control flow graph of the assembly code;
  each box is a _basic block_ (a sequence of instructions executed straight through) and each arrow is a jump (T = taken, F = not taken = falls through)
* a loop appears as a cycle in the graph


In [ ]:
import vis_asm
vis_asm.vis_asm_file("no_ilp.s")

# 7. SIMD
* SIMD instructions perform the same operation on several values at once
* as we saw in the SIMD notebook, we cannot always count on the compiler to vectorize code the way we want, so here we vectorize the code ourselves with a vector type
* in the following code, `floatv` is a vector of four floats (16 bytes) and `x` is of type `floatv`; `x = a * x + b` therefore performs four FMAs, one for each element


In [ ]:
%%writefile simd.cc
#include "ilp.h"

/* a vector of floats (4 floats on Grace) */
typedef float floatv __attribute__((vector_size(16), __may_alias__, aligned(sizeof(float))));
enum { L = sizeof(floatv) / sizeof(float) };

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
  floatv x = {i + 0.0f, i + 0.25f, i + 0.5f, i + 0.75f};
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      x = a * x + b;            // L (= 4) FMAs at once
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  R[i].x = (x[0] + x[1] + x[2] + x[3]) / 4.0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd.cc ilp_main.cc -o simd
#nvc++   -Wall -O3 -tp neoverse-v2 -mp=multicore -cuda simd.cc ilp_main.cc -o simd
#g++     -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd.cc ilp_main.cc -o simd

* run it on a single core


In [ ]:
%%bash
OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./simd 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* observe that the number of cycles per iteration is the same as before, even though each iteration now performs four FMAs
* put differently, the _throughput_, the number of operations executed per cycle (or per unit time), has quadrupled
* let's confirm that the loop indeed uses a SIMD FMA instruction


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd.cc -S

In [ ]:
%%bash
cat simd.s

* you will find instructions like `fmla v0.4s, v1.4s, v2.s[0]` in the innermost loop; `v0.4s` means register `v0` viewed as a vector of four single-precision floats (we will come back to the `mov` instruction accompanying each `fmla` later)

# 8. SIMD + ILP
* ILP increases when there are multiple instructions that are _independent_ of each other
* no processor can execute the following sequence faster than 4 cycles / FMA, because each instruction has to wait for the previous instruction to produce its result (i.e., it depends on the previous instruction)
```
x = a * x + b
x = a * x + b
x = a * x + b
  ...
```
* what a CPU _can_ do is execute something like the following as fast as 4 cycles / (2 FMAs)
```
x0 = a * x0 + b
x1 = a * x1 + b
x0 = a * x0 + b
x1 = a * x1 + b
x0 = a * x0 + b
x1 = a * x1 + b
  ...
```
* this is possible because instructions working on x0 do not depend on those working on x1 and vice versa

* the same applies to SIMD instructions; the following code updates two vector variables `x0` and `x1` (eight floats in total)


In [ ]:
%%writefile simd2.cc
#include "ilp.h"

/* a vector of floats (4 floats on Grace) */
typedef float floatv __attribute__((vector_size(16), __may_alias__, aligned(sizeof(float))));
enum { L = sizeof(floatv) / sizeof(float) };

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
  floatv x0 = {i + 0.0f, i + 0.125f, i + 0.25f, i + 0.375f};
  floatv x1 = {i + 0.5f, i + 0.625f, i + 0.75f, i + 0.875f};
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      x0 = a * x0 + b;
      x1 = a * x1 + b;
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  R[i].x = (x0[0] + x0[1] + x0[2] + x0[3] + x1[0] + x1[1] + x1[2] + x1[3]) / 8.0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd2.cc ilp_main.cc -o simd2
#nvc++   -Wall -O3 -tp neoverse-v2 -mp=multicore -cuda simd2.cc ilp_main.cc -o simd2
#g++     -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd2.cc ilp_main.cc -o simd2

* run it on a single core again


In [ ]:
%%bash
OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./simd2 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* observe that the number of cycles per iteration hardly changes again, while each iteration now performs eight FMAs (two SIMD FMA instructions)

# 9. Increase ILP further
* let's increase ILP further
* in the following code, you can configure the number of floats updated in the `k` loop by setting a preprocessor constant `C`; they are held in `C / L` vectors (`L` = 4), so `C` must be a multiple of 4, and there are `C / L` _independent_ SIMD FMA instructions in each iteration
* how far can it go?  Grace has four execution units for floating point and SIMD operations (named V0-V3 in the optimization guide), so it can execute up to four SIMD FMAs, i.e., sixteen FMAs on floats, per cycle


In [ ]:
%%writefile simd_ilp.cc
#include "ilp.h"

/* a vector of floats (4 floats on Grace) */
typedef float floatv __attribute__((vector_size(16), __may_alias__, aligned(sizeof(float))));
enum { L = sizeof(floatv) / sizeof(float) };

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
#ifndef C
#error "give -DC=xxx in the command line"
#endif
  static_assert(C % L == 0, "C must be a multiple of L (the number of floats in a vector)");
  // C floats in C / L vectors
  floatv x[C / L];
  for (int c = 0; c < C / L; c++) {
    for (int l = 0; l < L; l++) {
      x[c][l] = i + (c * L + l) / (float)C;
    }
  }
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      for (int c = 0; c < C / L; c++) {
        x[c] = a * x[c] + b;    // L FMAs at once
      }
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  double t = 0.0;
  for (int c = 0; c < C / L; c++) {
    for (int l = 0; l < L; l++) {
      t += x[c][l];
    }
  }
  R[i].x = t / C;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -DC=16 -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd_ilp.cc ilp_main.cc -o simd_ilp
#nvc++   -DC=16 -Wall -O3 -tp neoverse-v2 -mp=multicore -cuda simd_ilp.cc ilp_main.cc -o simd_ilp
#g++     -DC=16 -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd_ilp.cc ilp_main.cc -o simd_ilp

In [ ]:
%%bash
OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./simd_ilp 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* change `C` above and see how far you can go without noticeably increasing the cycles per iteration
* then change `C` systematically and graph the result
* the throughput in the graph is the number of FMAs on _floats_ per cycle, so an FMA instruction on a vector of four floats counts as four
* make sure to increase `C` to observe that the throughput initially rises and eventually plateaus at a certain point
* given that Grace can execute four SIMD FMAs per cycle, each on four floats, how far does it go compared to the ideal (16 FMAs per cycle)?


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
for L in 1; do
    for M in 100 ; do
        for N in 1000000; do
            for C in 4 8     ; do  # put values of C you want to experiment with (must be multiples of 4)
                clang++ -DC=${C} -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd_ilp.cc ilp_main.cc -o simd_ilp_${C} \
                    && (echo -n "L=${L} M=${M} N=${N} C=${C} ";
                        OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./simd_ilp_${C} ${L} ${M} ${N} | awk '{print $5}')
            done
        done
    done
done 

In [ ]:
# copy the result above below
DATA_STRING_SIMD = r"""
L=1 M=100 N=10000000 C=4 cycles_per_iter=4.000081
L=1 M=100 N=10000000 C=8 cycles_per_iter=4.000105
    ...
"""

In [ ]:
import vis_latency_throughput
vis_latency_throughput.vis(DATA_STRING_SIMD)

## 9-1. Why does it plateau so early?
* you will observe that the throughput plateaus at around 8 FMAs per cycle (2 SIMD FMA instructions per cycle), only half of the ideal (4 SIMD FMA instructions per cycle), and even goes down when `C` is increased further
* the reason is a bit complicated and somewhat particular to ARM instruction and what this code does
* let's look at the assembly code


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -DC=32 -Wall -O3 -mcpu=neoverse-v2 -fopenmp simd_ilp.cc -S

In [ ]:
%%bash
cat simd_ilp.s

* you will find the innermost loop looks something like this (details may differ)
```
mov     v0.16b, v25.16b
mov     v1.16b, v25.16b
    ...
fmla    v0.4s, v23.4s, v24.s[0]
fmla    v1.4s, v22.4s, v24.s[0]
    ...
```
* `fmla v0.4s, v23.4s, v24.s[0]` is a NEON FMA instruction, which computes `v0 += v23 * v24[0]` (`v24.s[0]` is the first float of `v24`, i.e., `a`); that is, the _addend_ (`v0`) is overwritten with the result
* our computation is `x = a * x + b`, in which the addend is `b`, a value that must be kept intact; the compiler therefore copies `b` (in `v25`) into a fresh register (`mov v0.16b, v25.16b`) before each `fmla`, which then becomes the new `x`
* these `mov` instructions are not free; `mov` and `fmla` appear to compete for the same resource that can execute 4 of them per cycle (the four SIMD execution units); half of their capacity is spent on `mov`, which is why the throughput plateaus at 2 SIMD FMA instructions per cycle
* note that the scalar FMA instruction you saw before (`fmadd s0, s9, s0, s8`) has a separate destination register (four operands in total), so it does not have this problem

## 9-2. Using SVE instructions
* recall that Grace supports another SIMD instruction set, SVE
* SVE has an FMA instruction that puts the result in the _multiplicand_ rather than the addend
```
fmad    z0.s, p0/m, z25.s, z24.s    // z0 = z0 * z25 + z24
```
  * `z0`, `z24`, ... are SVE vector registers (128 bits on Grace; `v0`, `v24`, ... are their lower halves)
  * `p0` is a _predicate register_ specifying which elements (lanes) to compute; here it has all lanes turned on
* this is exactly `x = x * a + b`, so no `mov` is necessary
* even with `-mcpu=neoverse-v2`, however, the compiler uses NEON instructions for operations on vector types like `floatv`
* to force it to use SVE instructions, we compute on an SVE type, `svfloat32_t` (a vector of floats in SVE), by casting a `floatv` value to it; the loop becomes
```
x[c] = a * (svfloat32_t)x[c] + b;
```
* this cast is allowed only with the `-msve-vector-bits=128` option, which tells the compiler the vector length of SVE is 128 bits, the same as `floatv`
  * the SVE instruction set itself does not fix the vector length (it can be anything from 128 to 2048 bits, depending on the processor), and this is why it is called the _Scalable_ Vector Extension
  * with this option, the generated code runs correctly only on processors whose SVE vector length is 128 bits, like Grace
* an alternative to the cast is to use an SVE intrinsic function corresponding to the `fmad` instruction, `svmad_f32_x`
```
x[c] = svmad_f32_x(svptrue_b32(), x[c], svdup_f32(a), svdup_f32(b));
```


In [ ]:
%%writefile simd_ilp_sve.cc
#include <arm_sve.h>
#include "ilp.h"

/* a vector of floats (4 floats on Grace) */
typedef float floatv __attribute__((vector_size(16), __may_alias__, aligned(sizeof(float))));
enum { L = sizeof(floatv) / sizeof(float) };

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
#ifndef C
#error "give -DC=xxx in the command line"
#endif
  static_assert(C % L == 0, "C must be a multiple of L (the number of floats in a vector)");
  // C floats in C / L vectors
  floatv x[C / L];
  for (int c = 0; c < C / L; c++) {
    for (int l = 0; l < L; l++) {
      x[c][l] = i + (c * L + l) / (float)C;
    }
  }
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      for (int c = 0; c < C / L; c++) {
        // compute on the SVE type, so that the compiler uses SVE instructions
        // (needs -msve-vector-bits=128 to allow this cast)
        x[c] = a * (svfloat32_t)x[c] + b;
      }
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  double t = 0.0;
  for (int c = 0; c < C / L; c++) {
    for (int l = 0; l < L; l++) {
      t += x[c][l];
    }
  }
  R[i].x = t / C;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -DC=32 -Wall -O3 -mcpu=neoverse-v2 -msve-vector-bits=128 -fopenmp simd_ilp_sve.cc -S

In [ ]:
%%bash
cat simd_ilp_sve.s

* confirm that the innermost loop now consists of `fmad` instructions without `mov`
* then change `C` systematically and graph the result


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
for L in 1; do
    for M in 100 ; do
        for N in 1000000; do
            for C in 4 8     ; do  # put values of C you want to experiment with (must be multiples of 4)
                clang++ -DC=${C} -Wall -O3 -mcpu=neoverse-v2 -msve-vector-bits=128 -fopenmp simd_ilp_sve.cc ilp_main.cc -o simd_ilp_sve_${C} \
                    && (echo -n "L=${L} M=${M} N=${N} C=${C} ";
                        OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./simd_ilp_sve_${C} ${L} ${M} ${N} | awk '{print $5}')
            done
        done
    done
done 

In [ ]:
# copy the result above below
DATA_STRING_SVE = r"""
L=1 M=100 N=10000000 C=4 cycles_per_iter=4.000084
L=1 M=100 N=10000000 C=8 cycles_per_iter=4.000095
    ...
"""

In [ ]:
import vis_latency_throughput
vis_latency_throughput.vis(DATA_STRING_SVE)

* how many independent FMAs (`C`) are necessary to reach the throughput limit (16 FMAs per cycle)?  think about it from the latency (4 cycles), the number of execution units (4) and the number of floats in a vector (4)
* when you go beyond it, you may observe that the throughput fluctuates between runs
  * look at the time stamps printed after `cycles_per_iter=...` (the cycle counter read every `N` iterations); the throughput of each interval is either 16 or 8 FMAs per cycle, as if the processor chose one of the two modes each time it enters the loop
  * we do not know why this happens
* also note that there are only 32 vector registers; with too large a `C`, `x` no longer fits in registers, and the loop must load and store some of them

# 10. GPU 
* what about GPUs?
* GPUs do not aggressively exploit ILP
* let's first see the case where we have almost no ILP
* the code can be exactly the same as in the CPU case; we merely have to change the compile options
  * `--offload-arch=sm_90` generates GPU code
  * `-foffload-lto` is needed because the GPU code spans two files (`iter_fun` in one and the target region calling it in `ilp_main.cc`); without it, clang (as of this writing) silently generates a wrong GPU version of `iter_fun`


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -O3 -mcpu=neoverse-v2 -fopenmp --offload-arch=sm_90 -foffload-lto no_ilp.cc ilp_main.cc -o no_ilp_gpu
#nvc++   -Wall -O3 -tp neoverse-v2 -mp=gpu -cuda no_ilp.cc ilp_main.cc -o no_ilp_gpu

* and run it with a single thread


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./no_ilp_gpu 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* this is the _latency_ of an FMA instruction on the NVIDIA GPU
* it is slightly longer than on the CPU in terms of the number of cycles; it is even longer in terms of absolute time (as the frequency of the H100 GPU is lower than that of Grace, which goes up to 3.4GHz)

* another piece of good news is that the same executable can actually run on CPUs too
* we merely have to disable offloading by setting an environment variable


In [ ]:
%%sub
# execute on CPU
OMP_TARGET_OFFLOAD=DISABLED OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./no_ilp_gpu 1 100 1000000 > a.dat
awk '{print $5}' a.dat

# 11. GPU with ILP?
* all in all, the GPU is slower than Grace when it comes to the latency of an FMA (roughly twice as slow in absolute time)
* but that's not the main point
* the main point is what happens if we expose more independent FMAs to a _single thread_ of the GPU
* we use the following scalar version, which updates `C` (scalar) variables


In [ ]:
%%writefile ilp.cc
#include "ilp.h"

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */

void iter_fun(float a, float b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
#ifndef C
#error "give -DC=xxx in the command line"
#endif
  float x[C];
  for (int c = 0; c < C; c++) {
    x[c] = i + c / (float)C;
  }
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting SM
  R[i].core[0] = get_core();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = get_clock();
    asm volatile("// ========== loop begins ==========");
    for (long k = 0; k < N; k++) {
      for (int c = 0; c < C; c++) {
        x[c] = a * x[c] + b;
      }
    }
    asm volatile("// ---------- loop ends ----------");
  }
  // record ending SM (must be = sm[0])
  R[i].core[1] = get_core();
  // record result, just so that the computation is not
  // eliminated by the compiler
  double t = 0.0;
  for (int c = 0; c < C; c++) {
    t += x[c];
  }
  R[i].x = t / C;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -DC=2 -Wall -O3 -mcpu=neoverse-v2 -fopenmp --offload-arch=sm_90 -foffload-lto ilp.cc ilp_main.cc -o ilp_gpu
#nvc++   -DC=2 -Wall -O3 -tp neoverse-v2 -mp=gpu -cuda ilp.cc ilp_main.cc -o ilp_gpu

* and run it with a single thread on the GPU


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./ilp_gpu 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* and on the CPU


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=DISABLED OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./ilp_gpu 1 100 1000000 > a.dat
awk '{print $5}' a.dat

* change the value of `C` and see what happens on the GPU

* then change `C` systematically and graph the result
* observe that the latency immediately starts increasing, and thus the throughput does not increase


In [ ]:
%%sub
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
for L in 1; do
    for M in 100 ; do
        for N in 1000000; do
            for C in 1 2    ; do  # put values of C you want to experiment with
                # choose the compiler and craft an appropriate command
                # line to produce exe file ilp_gpu_${C}
                #nvc++   -DC=${C} -Wall -O3 -tp neoverse-v2 -mp=gpu -cuda ilp.cc ilp_main.cc -o ilp_gpu_${C} \
                clang++ -DC=${C} -Wall -O3 -mcpu=neoverse-v2 -fopenmp --offload-arch=sm_90 -foffload-lto ilp.cc ilp_main.cc -o ilp_gpu_${C} \
                    && (echo -n "L=${L} M=${M} N=${N} C=${C} ";
                        OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 OMP_PROC_BIND=true ./ilp_gpu_${C} ${L} ${M} ${N} | awk '{print $5}')
            done
        done
    done
done 

In [ ]:
# copy the result above below
DATA_STRING_GPU = r"""
L=1 M=100 N=10000000 C=1 cycles_per_iter=2.128242
L=1 M=100 N=10000000 C=2 cycles_per_iter=2.126931
L=1 M=100 N=10000000 C=3 cycles_per_iter=2.125129
    ...
"""

In [ ]:
import vis_latency_throughput
vis_latency_throughput.vis(DATA_STRING_GPU)

* As you can see, performance of a single CUDA thread is very limited
* As you have seen in the previous exercise (`pd06_visualize`), a single SM is able to execute many threads simultaneously and maximum throughput is 128 single precision FMAs (4 warps) per cycle

# 12. Summary: CPU vs GPU
* _Both_ CPU and GPU impose roughly the same latency (four cycles) for single precision FMA (or 1/4 FMA per cycle)
* There is no way to _reduce latency_; the only way to increase performance is to increase _throughput_, by doing multiple independent computations simultaneously

* A single CPU _thread_ is able to increase parallelism by using SIMD and extracting instruction-level parallelism (ILP)
  * For single precision, 4x by SIMD
  * For FMAs, 16x by ILP (the maximum throughput is 4 FMAs per cycle, which is 16 x (1/4) FMA per cycle)
  * Together, on CPU, _a single thread_ is able to execute 16 single-precision FMAs (4 FMAs per cycle)
  * This is the performance limit of a _core_
  
* A CUDA thread has no SIMD; a single CUDA thread has only a very limited ILP
  * According to the experiment, a single CUDA thread is able to execute 0.75 FMAs per cycle
  * This is _not_ the performance limit of an _SM_, which can execute many threads simultaneously
  * even without using ILP at all (1/4 FMAs per CUDA core per cycle), you can simply create many (e.g., 512) threads on a single SM and easily reach the throughput limit (128 FMAs per cycle)
  
* On _CPU_, parallelism is about the combination of SIMD (4x), ILP (16x), and multiple cores (72)
* On _GPU_, parallelism is about the combination of a little bit of ILP (2-3), a large number of threads per SM (e.g., 1024), and multiple SMs (132)
* (numbers are for Grace and Hopper)

  